# Provenance Agent — Data Workflow (Quickstart)

This notebook mirrors `../workflow.ipynb` for the **data** side. Given a notebook, the data workflow detects which variables hold the datasets used for analysis (via an LLM) and generates dataset citations (BibTeX).

**Functions covered:**
- `detect_datasets` / `detect_datasets_in_notebook` — LLM detection of dataset variables
- `build_retrieval_cell` — per-tool citation-retrieval source
- `filter_datasets` — narrow to all / one tool / one variable
- `inject_retrieval_cells` — append retrieval cells via nbformat
- `generate_data_workflow` — end-to-end: detect → filter → inject → write

## Setup

Add `src/` to the path. This notebook lives in `notebooks/testing/`, so `src/` is two levels up.

In [7]:
import sys
sys.path.insert(0, '../../src')

from dataset_detection import detect_datasets, detect_datasets_in_notebook
from data_workflow import (
    build_retrieval_cell,
    filter_datasets,
    inject_retrieval_cells,
    generate_data_workflow,
)

## 1. `detect_datasets` — LLM detection of dataset variables

Detection is done by an LLM (Gemini via LangChain), not AST parsing. It traces the data flow to the **terminal** variable actually used for analysis and returns `[variable, tool]` pairs. Here it finds the filtered LiPDGraph result in the PaleoPCAlite example.

In [8]:
pairs = detect_datasets_in_notebook('paleoPCAlite.ipynb')
print(pairs)

[['filtered_df2', 'LiPDGraph']]


## 2. `build_retrieval_cell` — per-tool retrieval source

Each dataset gets a code cell that retrieves its BibTeX by reusing the already-loaded object in the live kernel (Approach C: `{var}.{method}`). LiPDGraph is special — its terminal variable is a DataFrame, so the cell converts it to a LiPD object first.

In [9]:
for var, tool in [('D', 'PyLiPD'), ('ds', 'PyleoTUPS'), ('filtered_df2', 'LiPDGraph')]:
    print(f'# --- {tool} ---')
    print(build_retrieval_cell(var, tool))
    print()

# --- PyLiPD ---
_bib_D, _ = D.get_bibtex(remote=True)
print("\n".join(_bib_D))

# --- PyleoTUPS ---
_pub_ds, _ = ds.get_publications()
print(_pub_ds.to_string(bib_format="bibtex"))

# --- LiPDGraph ---
from pylipd.lipd import LiPD
_names_filtered_df2 = filtered_df2["dataSetName"].unique().tolist()
_lipd_filtered_df2 = LiPD()
_lipd_filtered_df2.set_endpoint("https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic")
_lipd_filtered_df2.load_remote_datasets(_names_filtered_df2)
_bib_filtered_df2, _ = _lipd_filtered_df2.get_bibtex(remote=True)
print("\n".join(_bib_filtered_df2))



## 3. `filter_datasets` — all / one tool / one variable

Supports returning every dataset citation, or filtering to a single tool or variable.

In [10]:
sample = [['D', 'PyLiPD'], ['ds', 'PyleoTUPS'], ['filtered_df2', 'LiPDGraph']]
print('all:      ', filter_datasets(sample))
print('LiPDGraph:', filter_datasets(sample, tool='LiPDGraph'))
print('just ds:  ', filter_datasets(sample, variable='ds'))

all:       [['D', 'PyLiPD'], ['ds', 'PyleoTUPS'], ['filtered_df2', 'LiPDGraph']]
LiPDGraph: [['filtered_df2', 'LiPDGraph']]
just ds:   [['ds', 'PyleoTUPS']]


## 4. `inject_retrieval_cells` — append cells with nbformat

Appends one retrieval code cell per detected dataset to a notebook node.

In [11]:
import nbformat
demo = nbformat.v4.new_notebook()
inject_retrieval_cells(demo, [['filtered_df2', 'LiPDGraph']])
print(demo.cells[-1].source)

from pylipd.lipd import LiPD
_names_filtered_df2 = filtered_df2["dataSetName"].unique().tolist()
_lipd_filtered_df2 = LiPD()
_lipd_filtered_df2.set_endpoint("https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic")
_lipd_filtered_df2.load_remote_datasets(_names_filtered_df2)
_bib_filtered_df2, _ = _lipd_filtered_df2.get_bibtex(remote=True)
print("\n".join(_bib_filtered_df2))


## 5. `generate_data_workflow` — end to end

Detects datasets, optionally filters, injects the retrieval cells, and writes the notebook back. Here we write to a **copy** so the original is untouched. Open that copy and run the injected cell **in its live kernel** (where `filtered_df2` exists) to print the BibTeX.

Pass `tool=` or `variable=` to cite only part of the notebook, e.g. `generate_data_workflow('paleoPCAlite.ipynb', tool='LiPDGraph')`.

In [ ]:
pairs = generate_data_workflow(
    'paleoPCAlite.ipynb',
    output_path='paleoPCAlite_with_citations.ipynb',
)
print('injected cells for:', pairs)

---

## 6. The agent layer - three ways to call it

Sections 1-5 call the data workflow functions directly. The agent wraps that same work in three layers (mirroring `../workflow.ipynb`), shown lowest to highest.

**A note on mutation:** the data workflow injects retrieval cells *into the target notebook*. The direct tool takes an `output_path` so it can write to a copy, but `agent.run` and `%provenance` write **in place**. So the higher layers below run against a throwaway copy, `paleoPCAlite_demo.ipynb`, to leave the original untouched.

### The direct tool - `cite_data`

`orchestrator.cite_data` wraps `generate_data_workflow`. `output_path` writes the injected cells to a copy, so the source notebook is not modified. It returns the `[variable, tool]` pairs it injected.

In [ ]:
from orchestrator import cite_data

pairs = cite_data(
    'paleoPCAlite.ipynb',
    output_path='paleoPCAlite_with_citations.ipynb',
)
print('injected cells for:', pairs)

### The natural-language router - `agent.run`

`agent.run` routes 'cite the datasets' to `cite_data` with no `output_path`, so it writes in place. We point it at a fresh copy. The `result` is the list of injected `[variable, tool]` pairs - the citations themselves are produced when you open that copy and run the injected cells in its live kernel.

In [ ]:
import shutil
from agent import run

shutil.copy('paleoPCAlite.ipynb', 'paleoPCAlite_demo.ipynb')
for call in run('cite the datasets', 'paleoPCAlite_demo.ipynb'):
    print(call['name'], '->', call['result'])

### The `%provenance` magic

The notebook-native layer. It also writes in place, so we keep pointing it at the throwaway copy. After running, open `paleoPCAlite_demo.ipynb` and run the injected cells to print the BibTeX.

In [ ]:
import sys
sys.path.insert(0, '../../src')
%load_ext provenance

In [ ]:
%provenance_notebook paleoPCAlite_demo.ipynb

In [ ]:
%provenance cite the datasets